In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# Remove if you run local
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Remove if you run local
os.chdir("/kaggle/input/nyc-job-postings-with-esco-occupations-and-skills/")

In [ ]:
import pandas as pd
import random
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
skills = pd.read_excel('skills - 15000 sample.xlsx')
titles = pd.read_excel('titles.xlsx')
nyc_jobs = pd.read_excel('NYC Jobs (snapshot 2025-07-13) (anonymized).xlsx')

pay_cols = ['id', 'Salary Range From', 'Salary Range To', 'Salary Frequency']
nyc_pay  = nyc_jobs[pay_cols].copy()

# Merge occupation label + pay cols onto skills
df = (skills
      .merge(titles[['jobposting_id', 'title_esco_label']], on='jobposting_id', how='left')
      .merge(nyc_pay, left_on='jobposting_id', right_on='id', how='left')
      .drop(columns=['id'])  # keep if you need it later
     )

df.head()

## Top 10 ESCO Skills

In [ ]:
# Compute top 10 ESCO skills
top_skills = df['skill_esco_label'].value_counts().head(10)

max_count = top_skills.values.max()
xmax = max_count * 1.2

fig, ax = plt.subplots(figsize=(8, 5))

# Horizontal bars
bars = ax.barh(top_skills.index, top_skills.values, color="#F9A602")

for bar in bars:
    ax.text(
        bar.get_width() + xmax * 0.02,
        bar.get_y() + bar.get_height() / 2,
        f"{int(bar.get_width())}",
        va="center",
        ha="left",
        fontsize=9,
        clip_on=False
    )

# Styling
ax.set_xlim(0, xmax)
ax.set_title("Top 10 ESCO Skills in NYC Sample", fontsize=14, weight="bold")
ax.set_xlabel("Count of job postings mentioning skill")
ax.invert_yaxis()  # highest on top
ax.grid(axis="x", linestyle=":", alpha=0.3)

plt.tight_layout()
plt.show()


## Drill‑down: Occupation → Most Frequent Skill

### Simple approach 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Step 1: find top skill per occupation
pivot = (
    df
    .groupby(['title_esco_label', 'skill_esco_label'])
    .size()
    .reset_index(name='count')
)
top_skill_per_occ = (
    pivot
    .loc[pivot.groupby('title_esco_label')['count'].idxmax()]
    .sort_values(by='count', ascending=False)
    .head(20)
)

# Step 2: plot horizontal bars
fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(
    top_skill_per_occ['title_esco_label'],
    top_skill_per_occ['count'],
    color="#F9A602"
)

# Step 3: annotate each bar with “Skill (count)”
for bar, (_, row) in zip(bars, top_skill_per_occ.iterrows()):
    skill = row['skill_esco_label']
    count = row['count']
    ax.text(
        bar.get_width() + bar.get_width() * 0.02,
        bar.get_y() + bar.get_height() / 2,
        f"{skill} ({count})",
        va='center',
        ha='left',
        fontsize=9,
        clip_on=False
    )

# Styling
ax.set_title(
    "Most Frequent ESCO Skill by Occupation",
    fontsize=14, weight='bold'
)
ax.set_xlabel("Count of job postings mentioning skill")
ax.grid(axis='x', linestyle=':', alpha=0.3)
ax.invert_yaxis()  # highest on top
plt.tight_layout()
plt.show()


## Which Skills Really Define Each Occupation?

### On larger datasets, don’t stop at a single “top skill.” Pull the top 20 (or even 50) skills per occupation group—patterns in the long tail often reveal niche, high‑value capabilities.

In [ ]:
N = len(df)

# Pair, occ and global counts
pair = (df.groupby(['title_esco_label', 'skill_esco_label'])
          .size().rename('n_ik').reset_index())

occ  = df.groupby('title_esco_label').size().rename('n_i').reset_index()
skill = df.groupby('skill_esco_label').size().rename('n_k').reset_index()

pivot = (pair.merge(occ, on='title_esco_label')
              .merge(skill, on='skill_esco_label'))

# Shares
pivot['p_occ']    = pivot['n_ik'] / pivot['n_i']   # skill share inside this occupation
pivot['p_global'] = pivot['n_k']  / N              # skill share overall

# Distinctiveness metrics
eps = 1e-9
pivot['log_lift'] = np.log(pivot['p_occ'] + eps) - np.log(pivot['p_global'] + eps)
pivot['lift']     = pivot['p_occ'] / (pivot['p_global'] + eps)
pivot['tfidf']    = pivot['n_ik'] * np.log(N / (pivot['n_k'] + 1))

# Pick your metric
metric = 'log_lift'  # or 'tfidf' / 'lift'

top_skill_per_occ = (pivot.sort_values(metric, ascending=False)
                           .groupby('title_esco_label', as_index=False)
                           .head(1)
                           .sort_values(metric, ascending=False))

top_skill_per_occ.head(20)


## Show top 10 skills for a given occupation (or random if not provided)

In [ ]:
# Set occupation label here, or leave as None to pick randomly
occupation = None  # e.g., 'Community health workers', 'Education managers', 'ICT quality assurance manager'

if occupation is None:
    occupation = df['title_esco_label'].dropna().sample(1).iloc[0]

print(f"Top 10 skills for occupation: {occupation}")
skills_in_occ = df[df['title_esco_label'] == occupation]['skill_esco_label'].value_counts().head(10)
plt.figure(figsize=(8,5))
skills_in_occ.plot(kind='bar', color="#F9A602")
plt.title(f"Top 10 ESCO Skills for {occupation}", fontsize=14, weight="bold")
plt.ylabel("Count")
plt.xticks(rotation=45, ha="right")
plt.gca().grid(axis="x", linestyle=":", alpha=0.3)
plt.tight_layout()
plt.show()

## Ideas & Questions to Explore with This Data

### Pay & Occupations
- Which ESCO occupation *groups* have the highest median salary?  
- Are there outlier roles where salary is far above the group average? Why?

### Skills vs. Salary
- Which individual skills correlate with the top salary deciles?  
- Do transversal (soft) skills or technical (digital/green) skills predict pay better?

### Geography / Unit Differences
- How do skill demands differ for the *same occupation* across boroughs/agencies/divisions?  
- Are “green” or “digital” flags concentrated in certain departments?

### Trend & Forecasting (with bigger samples)
- Project forward: which emerging skills show the steepest growth curve (“skills of tomorrow”)?  
- Which skills are declining in frequency (still required but fading)?

### Skill Bundles & Co‑occurrence
- What skill clusters frequently appear together (network / association rules)?  
- Are there “hidden” prerequisite skill sets behind high‑demand roles?

### Hiring Friction Indicators
- Which occupations show many postings but low unique skill variety (possible hiring bottlenecks)?  
- Skills with high demand but low supply in postings history (gap analysis).

### Qualification Inflation
- Do junior roles list senior‑level skills?

### Curriculum & Training Signals
- Map top skills to university/bootcamp curricula—where are the mismatches?  
- Identify “adjacent” upskilling paths (skills that bridge lower‑paid to higher‑paid roles).

### ESG / Policy Angles
- Track the diffusion of “green” competencies outside environmental departments.  
- Which roles embed digital accessibility or AI‑governance skills?

### Robustness & Quality Checks
- Confidence score distributions: are some occupations systematically under‑ or over‑tagged?  
- Sensitivity: how do results change if we raise the skill fit‑score threshold?